# Post-hoc Amazon Eval Analysis

This notebook reads a completed `amazon_eval_*` directory from `eval_outputs/` and computes richer comparisons across `base`, `AS_IS`, and `AUTO` without rerunning inference.


In [1]:
import json
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

EVAL_DIR_OVERRIDE = None
SAVE_ANALYSIS_FILES = True
EXAMPLE_LIMIT = 10


def newest_eval_dir(root: Path):
    if not root.exists():
        return None
    candidates = [p for p in root.iterdir() if p.is_dir() and p.name.startswith("amazon_eval_")]
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)


candidate_roots = [
    Path("/workspace/eval_outputs"),
    Path("eval_outputs"),
    Path("analysis/eval_outputs"),
]

if EVAL_DIR_OVERRIDE:
    eval_dir = Path(EVAL_DIR_OVERRIDE)
else:
    eval_dir = next((d for d in (newest_eval_dir(root) for root in candidate_roots) if d is not None), None)

if eval_dir is None or not eval_dir.exists():
    raise FileNotFoundError(
        "No completed amazon_eval directory found. Set EVAL_DIR_OVERRIDE to an existing eval_outputs folder."
    )

print(f"Using eval dir: {eval_dir}")


Using eval dir: /workspace/eval_outputs/amazon_eval_20260320_095916


In [2]:
selection_path = eval_dir / "selection.json"
comparison_summary_path = eval_dir / "comparison_summary.csv"
output_paths = sorted(eval_dir.glob("*_outputs.csv"))

if not output_paths:
    raise FileNotFoundError(f"No *_outputs.csv files found in {eval_dir}")

selection = json.loads(selection_path.read_text()) if selection_path.exists() else {}
comparison_summary_df = pd.read_csv(comparison_summary_path) if comparison_summary_path.exists() else None
variant_frames = {
    p.name.replace("_outputs.csv", ""): pd.read_csv(p)
    for p in output_paths
}
variants = sorted(variant_frames)

display(Markdown("## Selected Run"))
display(pd.DataFrame([selection]))

display(Markdown("## Existing Aggregate Summary"))
if comparison_summary_df is not None:
    display(comparison_summary_df)
else:
    print("comparison_summary.csv not found")

display(Markdown("## Loaded Variant Files"))
display(
    pd.DataFrame(
        [
            {
                "variant": name,
                "rows": len(df),
                "columns": ", ".join(df.columns),
            }
            for name, df in variant_frames.items()
        ]
    )
)


## Selected Run

,base_model_id,model_source,checkpoint_tag,max_rows,temperature,top_p,max_new_tokens,amazon_csv_path,as_is_run_dir,auto_run_dir
0,google/gemma-3-12b-it,/workspace/.hf/hub/models--google--gemma-3-12b...,final,50,0.0,1.0,96,/workspace/Amazon_filter_reviewNum_876 - Amazo...,/workspace/outputs/run_20260311_101734,/workspace/outputs/run_20260311_114945


## Existing Aggregate Summary

,variant,n,empty_rate,mean_word_count,mean_sentence_count,mean_keyword_overlap,mean_forbidden_hits,mean_prompt_leak_hits,delta_vs_base__mean_word_count,delta_vs_base__mean_sentence_count,delta_vs_base__mean_keyword_overlap,delta_vs_base__mean_forbidden_hits,delta_vs_base__mean_prompt_leak_hits
0,as_is,50,0.0,45.0,2.1,0.073869,0.86,0.0,0.0,0.0,0.0,0.0,0.0
1,auto,50,0.0,45.0,2.1,0.073869,0.86,0.0,0.0,0.0,0.0,0.0,0.0
2,base,50,0.0,45.0,2.1,0.073869,0.86,0.0,0.0,0.0,0.0,0.0,0.0


## Loaded Variant Files

,variant,rows,columns
0,as_is,50,"variant, ASIN, ASIN_Title, n_reviews, avg_rati..."
1,auto,50,"variant, ASIN, ASIN_Title, n_reviews, avg_rati..."
2,base,50,"variant, ASIN, ASIN_Title, n_reviews, avg_rati..."


In [3]:
summary_col = "summary"
required_cols = {summary_col, "word_count", "sentence_count", "keyword_overlap", "forbidden_hits", "prompt_leak_hits"}
missing = {
    name: sorted(required_cols - set(df.columns))
    for name, df in variant_frames.items()
    if required_cols - set(df.columns)
}
if missing:
    raise ValueError(f"Missing expected columns: {missing}")


def safe_ratio(a: str, b: str) -> float:
    if a == b:
        return 1.0
    return SequenceMatcher(None, a, b).ratio()


pair_rows = []
for i, left in enumerate(variants):
    left_df = variant_frames[left]
    left_summaries = left_df[summary_col].fillna("").astype(str)
    for right in variants[i:]:
        right_df = variant_frames[right]
        right_summaries = right_df[summary_col].fillna("").astype(str)
        exact = left_summaries == right_summaries
        similarity = [safe_ratio(a, b) for a, b in zip(left_summaries, right_summaries)]
        pair_rows.append(
            {
                "left_variant": left,
                "right_variant": right,
                "n_rows": int(len(left_df)),
                "exact_match_rate": float(exact.mean()),
                "disagreement_rows": int((~exact).sum()),
                "mean_sequence_similarity": float(np.mean(similarity)),
                "mean_abs_word_count_delta": float(
                    np.mean(np.abs(left_df["word_count"] - right_df["word_count"]))
                ),
                "mean_abs_keyword_overlap_delta": float(
                    np.mean(np.abs(left_df["keyword_overlap"] - right_df["keyword_overlap"]))
                ),
                "mean_abs_forbidden_hits_delta": float(
                    np.mean(np.abs(left_df["forbidden_hits"] - right_df["forbidden_hits"]))
                ),
            }
        )

pairwise_df = pd.DataFrame(pair_rows)
exact_match_matrix = pairwise_df.pivot(index="left_variant", columns="right_variant", values="exact_match_rate")
similarity_matrix = pairwise_df.pivot(index="left_variant", columns="right_variant", values="mean_sequence_similarity")

display(Markdown("## Pairwise Summary Comparison"))
display(pairwise_df)

display(Markdown("## Exact Match Matrix"))
display(exact_match_matrix)

display(Markdown("## Mean Sequence Similarity Matrix"))
display(similarity_matrix)


## Pairwise Summary Comparison

,left_variant,right_variant,n_rows,exact_match_rate,disagreement_rows,mean_sequence_similarity,mean_abs_word_count_delta,mean_abs_keyword_overlap_delta,mean_abs_forbidden_hits_delta
0,as_is,as_is,50,1.0,0,1.0,0.0,0.0,0.0
1,as_is,auto,50,1.0,0,1.0,0.0,0.0,0.0
2,as_is,base,50,1.0,0,1.0,0.0,0.0,0.0
3,auto,auto,50,1.0,0,1.0,0.0,0.0,0.0
4,auto,base,50,1.0,0,1.0,0.0,0.0,0.0
5,base,base,50,1.0,0,1.0,0.0,0.0,0.0


## Exact Match Matrix

right_variant,as_is,auto,base
left_variant,,,
as_is,1.0,1.0,1.0
auto,NaN,1.0,1.0
base,NaN,NaN,1.0


## Mean Sequence Similarity Matrix

right_variant,as_is,auto,base
left_variant,,,
as_is,1.0,1.0,1.0
auto,NaN,1.0,1.0
base,NaN,NaN,1.0


In [4]:
anchor = "base" if "base" in variants else variants[0]
anchor_df = variant_frames[anchor]
id_cols = [c for c in ["ASIN", "ASIN_Title", "n_reviews", "avg_rating"] if c in anchor_df.columns]

row_compare_df = anchor_df[id_cols].copy()
summary_table = pd.DataFrame(index=anchor_df.index)

for variant in variants:
    df = variant_frames[variant]
    row_compare_df[f"{variant}_summary"] = df[summary_col].fillna("").astype(str)
    row_compare_df[f"{variant}_word_count"] = df["word_count"]
    row_compare_df[f"{variant}_keyword_overlap"] = df["keyword_overlap"]
    row_compare_df[f"{variant}_forbidden_hits"] = df["forbidden_hits"]
    row_compare_df[f"{variant}_prompt_leak_hits"] = df["prompt_leak_hits"]
    summary_table[variant] = df[summary_col].fillna("").astype(str)

row_compare_df["n_unique_summaries"] = summary_table.nunique(axis=1)
row_compare_df["all_equal"] = row_compare_df["n_unique_summaries"] == 1

for variant in variants:
    if variant == anchor:
        continue
    variant_summary = variant_frames[variant][summary_col].fillna("").astype(str)
    anchor_summary = anchor_df[summary_col].fillna("").astype(str)
    row_compare_df[f"{variant}_matches_{anchor}"] = variant_summary == anchor_summary
    row_compare_df[f"{variant}_similarity_to_{anchor}"] = [
        safe_ratio(a, b) for a, b in zip(anchor_summary, variant_summary)
    ]

disagreement_df = row_compare_df.loc[~row_compare_df["all_equal"]].copy()

display(Markdown("## Row-level Agreement Overview"))
overview_df = pd.DataFrame(
    [
        {
            "rows": int(len(row_compare_df)),
            "rows_with_any_difference": int((~row_compare_df["all_equal"]).sum()),
            "rows_identical_across_all_variants": int(row_compare_df["all_equal"].sum()),
            "anchor_variant": anchor,
        }
    ]
)
display(overview_df)

display(Markdown("## Unique Summary Count Per Row"))
display(row_compare_df[[*id_cols, "n_unique_summaries", "all_equal"]].head(15))

display(Markdown("## Example Disagreements"))
if disagreement_df.empty:
    print("No row-level disagreements were found. All saved summaries are identical across variants for this eval run.")
else:
    example_cols = id_cols + [c for c in row_compare_df.columns if c.endswith("_summary")]
    display(disagreement_df[example_cols].head(EXAMPLE_LIMIT))


## Row-level Agreement Overview

,rows,rows_with_any_difference,rows_identical_across_all_variants,anchor_variant
0,50,0,50,base


## Unique Summary Count Per Row

,ASIN,ASIN_Title,n_reviews,avg_rating,n_unique_summaries,all_equal
0,B004GJXE0E,"Farberware Bamboo Cutting Board, Set of 3 Asso...",14,4.285714,1,True
1,B018JMMCLU,HOUSE DAY Black Magic Hangers Space Saving Clo...,15,4.533333,1,True
2,B01MZ14LJV,Kitchen Gizmo Snap N Strain Pot Strainer and P...,17,4.764706,1,True
3,B06VWSC77W,Steel Paper Towel Holder by Home Basics (Turqu...,13,4.461538,1,True
4,B06XCXQFXM,"Pineapple Corer, Upgraded, Reinforced, Thicker...",16,4.875000,1,True
5,B06XQJN6F8,YOO.MEE Shut-Off Brass Ball Valve 1/2 Standard...,13,5.000000,1,True
6,B0775GR18G,AUOON Clip On Strainer Silicone for All Pots a...,14,4.857143,1,True
7,B078C7DBMT,Barnyard Designs One Small Positive Thought Wo...,16,4.875000,1,True
8,B07LFJ6RV4,The Grip Jar Opener | Original Under Cabinet J...,14,4.928571,1,True
9,B08887T7M5,"QiMH Collapsible Colander and Strainer, 5 Quar...",16,4.500000,1,True


## Example Disagreements

No row-level disagreements were found. All saved summaries are identical across variants for this eval run.


In [5]:
variant_uniques = []
for variant, df in variant_frames.items():
    summaries = df[summary_col].fillna("").astype(str)
    variant_uniques.append(
        {
            "variant": variant,
            "rows": int(len(df)),
            "unique_summary_count": int(summaries.nunique()),
            "duplicate_summary_rows": int(len(df) - summaries.nunique()),
            "mean_word_count": float(df["word_count"].mean()),
            "mean_keyword_overlap": float(df["keyword_overlap"].mean()),
            "mean_forbidden_hits": float(df["forbidden_hits"].mean()),
            "mean_prompt_leak_hits": float(df["prompt_leak_hits"].mean()),
        }
    )

variant_unique_df = pd.DataFrame(variant_uniques)

posthoc_report = {
    "eval_dir": str(eval_dir),
    "variants": variants,
    "rows": int(len(anchor_df)),
    "rows_with_any_difference": int((~row_compare_df["all_equal"]).sum()),
    "all_variants_identical": bool(row_compare_df["all_equal"].all()),
    "exact_match_pairs": pairwise_df[["left_variant", "right_variant", "exact_match_rate", "disagreement_rows"]].to_dict("records"),
}

display(Markdown("## Variant-level Output Diversity"))
display(variant_unique_df)

display(Markdown("## Post-hoc Conclusion"))
if posthoc_report["all_variants_identical"]:
    print(
        "For this completed Amazon eval run, base, AS_IS, and AUTO produced identical saved summaries on every evaluated row. "
        "This means there is no observable output-level transfer effect in this benchmark under the current prompts, sample, and decoding settings."
    )
else:
    print(
        f"Found {posthoc_report['rows_with_any_difference']} rows with variant disagreements out of {posthoc_report['rows']} total rows."
    )

if SAVE_ANALYSIS_FILES:
    pairwise_df.to_csv(eval_dir / "posthoc_pairwise_comparison.csv", index=False)
    exact_match_matrix.to_csv(eval_dir / "posthoc_exact_match_matrix.csv")
    similarity_matrix.to_csv(eval_dir / "posthoc_similarity_matrix.csv")
    row_compare_df.to_csv(eval_dir / "posthoc_row_level_comparison.csv", index=False)
    variant_unique_df.to_csv(eval_dir / "posthoc_variant_diversity.csv", index=False)
    (eval_dir / "posthoc_report.json").write_text(json.dumps(posthoc_report, indent=2))
    print(f"Saved post-hoc analysis files into: {eval_dir}")


## Variant-level Output Diversity

,variant,rows,unique_summary_count,duplicate_summary_rows,mean_word_count,mean_keyword_overlap,mean_forbidden_hits,mean_prompt_leak_hits
0,as_is,50,50,0,45.0,0.073869,0.86,0.0
1,auto,50,50,0,45.0,0.073869,0.86,0.0
2,base,50,50,0,45.0,0.073869,0.86,0.0


## Post-hoc Conclusion

For this completed Amazon eval run, base, AS_IS, and AUTO produced identical saved summaries on every evaluated row. This means there is no observable output-level transfer effect in this benchmark under the current prompts, sample, and decoding settings.
Saved post-hoc analysis files into: /workspace/eval_outputs/amazon_eval_20260320_095916
